In [3]:
import tensorflow as tf
import ultralytics
from ultralytics import YOLO
import os

gpu_tf = tf.config.list_physical_devices('GPU')
print(f"GPU TensorFlow: {gpu_tf}")
ultralytics.utils.checks.check_yolo()

Ultralytics 8.4.48 🚀 Python-3.11.15 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
Setup complete ✅ (12 CPUs, 7.4 GB RAM, 112.0/1006.9 GB disk)


In [4]:
model = YOLO('yolov8n-cls.pt')

dataset_path = '../SkinDisease'

In [5]:
results = model.train(
    data=dataset_path, 
    epochs=200,         
    patience=15,        
    imgsz=224,         
    batch=32,          
    device=0,          
    project='tubes_pasd', 
    name='skin_disease_classification',
    workers=2
)

New https://pypi.org/project/ultralytics/8.4.60 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.48 🚀 Python-3.11.15 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/petrichor/projects/skin-detection/dataset2/SkinDisease, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-c

In [7]:
best_model_path = '/home/petrichor/projects/skin-detection/notebook2/runs/classify/tubes_pasd/skin_disease_classification/weights/best.pt'
yolo_model = YOLO(best_model_path)

metrics = yolo_model.val(
    data=dataset_path, 
    split='test', 
    imgsz=224,
    device=0
)

accuracy = metrics.top1 * 100
print("\n" + "="*40)
print(f"Test Accuracy (22 Classes): {accuracy:.2f}%")
print("="*40)

Ultralytics 8.4.48 🚀 Python-3.11.15 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
YOLOv8n-cls summary (fused): 30 layers, 1,463,062 parameters, 0 gradients, 3.3 GFLOPs
train: /home/petrichor/projects/skin-detection/dataset2/SkinDisease/train... found 13898 images in 22 classes ✅ 
val: None...
test: /home/petrichor/projects/skin-detection/dataset2/SkinDisease/test... found 1546 images in 22 classes ✅ 
test: Fast image access ✅ (ping: 0.1±0.0 ms, read: 47.4±16.6 MB/s, size: 15.4 KB)
test: Scanning /home/petrichor/projects/skin-detection/dataset2/SkinDisease/test... 1546 images, 0 corrupt: 100% ━━━━━━━━━━━━ 1546/1546 49.9Mit/s 0.0s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 97/97 24.0it/s 4.0s0.1s
                   all      0.751      0.953
Speed: 0.2ms preprocess, 1.5ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to /home/petrichor/projects/skin-detection/notebook2/runs/classify/val

Test Accuracy (22 Classes): 75.10%


In [12]:
model = YOLO('/home/petrichor/projects/skin-detection/notebook2/runs/classify/tubes_pasd/skin_disease_classification/weights/best.pt')
test1 = '/home/petrichor/projects/skin-detection/download (2).jpeg'
test2 = '/home/petrichor/projects/skin-detection/download.jpeg'
test3 = '/home/petrichor/projects/skin-detection/images.jpeg'
test4 = '/home/petrichor/projects/skin-detection/download (1).jpeg'

result1 = model.predict(source=test1, save=True)
result2 = model.predict(source=test2, save=True)
result3 = model.predict(source=test3, save=True)
result4 = model.predict(source=test4, save=True)


image 1/1 /home/petrichor/projects/skin-detection/download (2).jpeg: 224x224 Vitiligo 1.00, DrugEruption 0.00, Unknown_Normal 0.00, Bullous 0.00, Actinic_Keratosis 0.00, 7.2ms
Speed: 2.7ms preprocess, 7.2ms inference, 0.1ms postprocess per image at shape (1, 3, 224, 224)
Results saved to /home/petrichor/projects/skin-detection/notebook2/runs/classify/predict-2

image 1/1 /home/petrichor/projects/skin-detection/download.jpeg: 224x224 Rosacea 0.76, Actinic_Keratosis 0.22, DrugEruption 0.02, Acne 0.00, Bullous 0.00, 23.2ms
Speed: 2.0ms preprocess, 23.2ms inference, 0.1ms postprocess per image at shape (1, 3, 224, 224)
Results saved to /home/petrichor/projects/skin-detection/notebook2/runs/classify/predict-2

image 1/1 /home/petrichor/projects/skin-detection/images.jpeg: 224x224 Actinic_Keratosis 0.27, Benign_tumors 0.23, SkinCancer 0.21, Bullous 0.12, DrugEruption 0.06, 28.4ms
Speed: 2.1ms preprocess, 28.4ms inference, 0.1ms postprocess per image at shape (1, 3, 224, 224)
Results saved t

In [13]:
for r in result1:
    top_class = r.names[r.probs.top1]
    confidence = r.probs.top1conf.item()*100
    print(f"Confidence {confidence:.2f}%")
    print(f"Disease {top_class.upper()}")

Confidence 100.00%
Disease VITILIGO


In [14]:
for r in result2:
    top_class = r.names[r.probs.top1]
    confidence = r.probs.top1conf.item()*100
    print(f"Confidence {confidence:.2f}%")
    print(f"Disease {top_class.upper()}")

Confidence 75.69%
Disease ROSACEA


In [15]:
for r in result3:
    top_class = r.names[r.probs.top1]
    confidence = r.probs.top1conf.item()*100
    print(f"Confidence {confidence:.2f}%")
    print(f"Disease {top_class.upper()}")

Confidence 26.56%
Disease ACTINIC_KERATOSIS


In [16]:
for r in result4:
    top_class = r.names[r.probs.top1]
    confidence = r.probs.top1conf.item()*100
    print(f"Confidence {confidence:.2f}%")
    print(f"Disease {top_class.upper()}")

Confidence 99.84%
Disease VITILIGO
